In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np
import joblib

import nltk
from nltk import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

In [2]:
nltk.download('punkt')
nltk.download('stopwords')

# 2. Load the Spanish stop words
stop_words_es = set(stopwords.words('spanish'))

[nltk_data] Downloading package punkt to /Users/samir/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/samir/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Preparación de datos

In [3]:
data = pd.read_excel('datos_miniproyecto2.xlsx')

In [4]:
data.head()

,textos,ODS
0,"""Aprendizaje"" y ""educación"" se consideran sinó...",4
1,No dejar clara la naturaleza de estos riesgos ...,6
2,"Como resultado, un mayor y mejorado acceso al ...",13
3,Con el Congreso firmemente en control de la ju...,16
4,"Luego, dos secciones finales analizan las impl...",5


In [5]:
data.duplicated().sum()

np.int64(0)

In [6]:
data.isna().sum()

textos    0
ODS       0
dtype: int64

In [7]:
X = data['textos']
y = data['ODS']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
# Pre-load stopwords to enable parallel usage
_stop_words_es = set(stopwords.words("spanish"))
_tokenizer = RegexpTokenizer(r"\w+")
_stemmer = PorterStemmer()

def text_preprocess(text):
    tokens = _tokenizer.tokenize(text)
    tokens = [word for word in tokens if word not in _stop_words_es]
    tokens = [_stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

In [9]:
vectorizer = TfidfVectorizer(preprocessor=text_preprocess)

In [10]:
tsvd = TruncatedSVD(n_components=10, random_state=42)

# Creación del pipeline y modelo de clasificación

In [11]:
model = LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced')

In [12]:
steps = [
    ("vectorizer", vectorizer),
    ("dimred", tsvd),
    ("model", model),
]
pipeline = Pipeline(steps)

In [13]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__max_iter': [500, 1000]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="accuracy", n_jobs=-1, verbose=2)

In [14]:
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
[CV] END ................model__C=0.01, model__max_iter=1000; total time=   3.5s
[CV] END .................model__C=0.01, model__max_iter=500; total time=   3.6s
[CV] END .................model__C=0.01, model__max_iter=500; total time=   3.7s
[CV] END ................model__C=0.01, model__max_iter=1000; total time=   3.7s
[CV] END ................model__C=0.01, model__max_iter=1000; total time=   3.7s
[CV] END ................model__C=0.01, model__max_iter=1000; total time=   3.7s
[CV] END ..................model__C=0.1, model__max_iter=500; total time=   3.8s
[CV] END .................model__C=0.01, model__max_iter=500; total time=   3.8s
[CV] END ................model__C=0.01, model__max_iter=1000; total time=   3.9s
[CV] END .................model__C=0.01, model__max_iter=500; total time=   3.9s
[CV] END ..................model__C=0.1, model__max_iter=500; total time=   3.9s
[CV] END .................model__C=0.01, model__m

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step..._iter=1000))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__C': [0.01, 0.1, ...], 'model__max_iter': [500, 1000]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see ho

In [15]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

In [16]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           1       0.85      0.76      0.80        90
           2       0.33      0.45      0.38        71
           3       0.88      0.80      0.84       178
           4       0.93      0.93      0.93       200
           5       0.93      0.85      0.89       232
           6       0.91      0.90      0.91       135
           7       0.91      0.78      0.84       164
           8       0.52      0.58      0.55        86
           9       0.23      0.33      0.27        55
          10       0.44      0.52      0.47        60
          11       0.52      0.36      0.42       143
          12       0.25      0.30      0.27        64
          13       0.56      0.69      0.62       102
          14       0.21      0.20      0.21        64
          15       0.37      0.40      0.38        81
          16       0.89      0.93      0.91       207

    accuracy                           0.70      1932
   macro avg       0.61   

In [17]:
joblib.dump(best_model,'model/miniproject2-model.joblib')

['model/miniproject2-model.joblib']

# Tópicos

In [18]:
vectorizer_output = vectorizer.fit_transform(X)
vectorizer.get_feature_names_out()

array(['00', '000', '0000002', ..., 'útero', 'útil', 'útile'],
      shape=(29799,), dtype=object)

In [19]:
tsvd_output = tsvd.fit_transform(vectorizer_output)
tsvd_output

array([[ 0.09316816,  0.01770463, -0.00493935, ...,  0.0204952 ,
         0.00454595,  0.0139404 ],
       [ 0.21269737, -0.26595594,  0.19288645, ...,  0.07717285,
         0.07350526, -0.02865758],
       [ 0.1525586 , -0.09539047,  0.0335559 , ..., -0.08781669,
        -0.03574522,  0.01814259],
       ...,
       [ 0.1119819 , -0.01988609,  0.00232831, ...,  0.0154158 ,
         0.05104183, -0.05608909],
       [ 0.16373777, -0.08872856,  0.05549061, ..., -0.03220084,
        -0.007294  , -0.01089098],
       [ 0.10709916,  0.00285432, -0.01299011, ...,  0.00912058,
         0.01615777,  0.00823198]], shape=(9656, 10))

In [20]:
vocabulario = vectorizer.get_feature_names_out()
loadings = tsvd.components_
# --- [PASO 3] Identificar las palabras con mayor peso para 5 componentes ---
n_palabras_por_topico = 10  # Número de palabras principales a mostrar
componentes_a_mostrar = 5   # Al menos 5 componentes

print("=== EXTRACCIÓN DE TÓPICOS (PALABRAS CON MAYOR PESO) ===")
for i in range(componentes_a_mostrar):
    # Obtener los índices de los pesos ordenados de menor a mayor, y tomar los últimos (los más grandes)
    indices_palabras_top = np.argsort(loadings[i])[::-1][:n_palabras_por_topico]

    # Mapear esos índices a las palabras reales
    palabras_top = [vocabulario[idx] for idx in indices_palabras_top]
    pesos_top = [loadings[i][idx] for idx in indices_palabras_top]

    print(f"\n🔹 Componente / Tópico {i + 1}:")
    for palabra, peso in zip(palabras_top, pesos_top):
        print(f"  - {palabra}: {peso:.4f}")

=== EXTRACCIÓN DE TÓPICOS (PALABRAS CON MAYOR PESO) ===

🔹 Componente / Tópico 1:
  - mujer: 0.1578
  - la: 0.1546
  - país: 0.1409
  - política: 0.1338
  - agua: 0.1303
  - desarrollo: 0.1155
  - el: 0.1136
  - derecho: 0.1134
  - en: 0.1131
  - nivel: 0.1011

🔹 Componente / Tópico 2:
  - mujer: 0.4580
  - derecho: 0.3364
  - género: 0.2430
  - hombr: 0.1586
  - humano: 0.1259
  - igualdad: 0.1197
  - trabajo: 0.0929
  - internacion: 0.0839
  - violencia: 0.0808
  - artículo: 0.0654

🔹 Componente / Tópico 3:
  - derecho: 0.5420
  - agua: 0.2360
  - humano: 0.2163
  - internacion: 0.1850
  - artículo: 0.1275
  - ley: 0.0886
  - penal: 0.0764
  - internacional: 0.0740
  - est: 0.0727
  - tratado: 0.0637

🔹 Componente / Tópico 4:
  - agua: 0.5441
  - mujer: 0.4587
  - género: 0.2310
  - hombr: 0.1644
  - igualdad: 0.1000
  - subterránea: 0.0814
  - residual: 0.0601
  - hídrico: 0.0550
  - violencia: 0.0538
  - riego: 0.0465

🔹 Componente / Tópico 5:
  - pobreza: 0.3302
  - ingreso: 0.219